In [ ]:
%pip install -q huggingface_hub pyarrow pandas matplotlib requests numpy scipy

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
ASSET               = "BTCUSDT"              # uppercase asset symbol
LOOK_BACK           = 1440                   # number of past candles (includes last_candle)
LOOK_AHEAD          = 240                    # number of future candles
DATETIME            = "2025-12-12 20:00:00" # current_time: moment after last_candle closes (UTC)
WINDOW_MODE         = "exclusive"            # "exclusive" | "inclusive"
NORM_MODE           = "clip"                 # "clip" | "tanh"
K                   = 100                    # k=100  →  ratio 0.01 maps to 1.0

BINS_COUNT          = 200                    # histogram bins for KDE (spans [-1, 1])
BANDWIDTH           = 5                      # kernel half-width in bins
KERNEL              = "Triangular"           # "Triangular" | "Epanechnikov" | "Uniform"

PLOT_VOLUME_LOGARITHM       = True           # if True draw log(norm_vol) instead of norm_vol
KDE_IGNORE_BORDERS          = True           # if True exclude clipped prices (±1.0) from KDE histogram
DELTA_VOLUME_RATIO_PERIOD   = 60             # SMA period for delta-volume ratio oscillator

In [ ]:
import io, re, requests
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import HfApi
from datetime import datetime, timedelta, timezone

REPO    = "https://huggingface.co/datasets/payamdavaee/candles/resolve/main"
REPO_ID = "payamdavaee/candles"

def load_range(asset, start, end, columns=None):
    api      = HfApi()
    files    = api.list_repo_files(REPO_ID, repo_type="dataset")
    asset_lc = asset.lower()
    pattern  = re.compile(rf"^{asset_lc}/{asset_lc}-1m-(\d{{4}})-(\d{{2}})\.parquet$")
    start_ts = int(datetime.fromisoformat(start).replace(tzinfo=timezone.utc).timestamp() * 1000)
    end_ts   = int(datetime.fromisoformat(end  ).replace(tzinfo=timezone.utc).timestamp() * 1000) + 86_400_000
    frames   = []
    for f in sorted(files):
        m = pattern.match(f)
        if not m:
            continue
        resp = requests.get(f"{REPO}/{f}", timeout=60)
        resp.raise_for_status()
        tbl  = pq.read_table(io.BytesIO(resp.content), columns=columns or None)
        df   = tbl.to_pandas()
        ts   = df["ts"].astype("int64")
        df   = df[(ts >= start_ts) & (ts < end_ts)]
        if not df.empty:
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames).sort_values("ts").reset_index(drop=True)


# ── Derive window boundaries from DATETIME ────────────────────────────────────
current_time     = datetime.fromisoformat(DATETIME).replace(tzinfo=timezone.utc)
last_candle_time = current_time - timedelta(minutes=1)
lb_start_time    = last_candle_time - timedelta(minutes=LOOK_BACK - 1)
lb_end_time      = last_candle_time
la_start_time    = current_time
la_end_time      = current_time + timedelta(minutes=LOOK_AHEAD - 1)

load_start = lb_start_time.strftime("%Y-%m-%d")
load_end   = (la_end_time + timedelta(days=1)).strftime("%Y-%m-%d")

print(f"Loading {ASSET} from {load_start} to {load_end} ...")
df_all = load_range(ASSET, load_start, load_end, columns=["ts", "vwap", "v", "vb", "vs"])
print(f"Loaded {len(df_all):,} candles")

ts_int      = df_all["ts"].astype("int64")
lb_start_ms = int(lb_start_time.timestamp() * 1000)
lb_end_ms   = int(lb_end_time.timestamp()   * 1000)
la_start_ms = int(la_start_time.timestamp() * 1000)
la_end_ms   = int(la_end_time.timestamp()   * 1000)

df_lb = df_all[(ts_int >= lb_start_ms) & (ts_int <= lb_end_ms)].reset_index(drop=True)
df_la = df_all[(ts_int >= la_start_ms) & (ts_int <= la_end_ms)].reset_index(drop=True)

assert len(df_lb) == LOOK_BACK,  f"Expected {LOOK_BACK} look-back candles, got {len(df_lb)}"
assert len(df_la) == LOOK_AHEAD, f"Expected {LOOK_AHEAD} look-ahead candles, got {len(df_la)}"

# ── Daily volume normalization ────────────────────────────────────────────────
v_daily_avg = df_lb["v"].mean()
v_lb_norm   = df_lb["v"].values / v_daily_avg
v_la_norm   = df_la["v"].values / v_daily_avg

# ── Delta-volume ratio  DVR = (vb - vs) / (vb + vs) ──────────────────────────
# vb + vs = v (by definition), so denominator is always > 0 when there is volume
def delta_volume_ratio(df):
    vb  = df["vb"].values
    vs  = df["vs"].values
    denom = vb + vs
    return np.where(denom > 0, (vb - vs) / denom, 0.0)

def trailing_sma(arr, period):
    """Trailing SMA; first period-1 values are NaN."""
    kernel = np.ones(period) / period
    valid  = np.convolve(arr, kernel, mode="valid")   # length = len(arr) - period + 1
    return np.concatenate([np.full(period - 1, np.nan), valid])

dvr_lb    = delta_volume_ratio(df_lb)
dvr_la    = delta_volume_ratio(df_la)
dvr_lb_ma = trailing_sma(dvr_lb, DELTA_VOLUME_RATIO_PERIOD)
dvr_la_ma = trailing_sma(dvr_la, DELTA_VOLUME_RATIO_PERIOD)

print(f"Look-back  : {len(df_lb)} candles  ✓")
print(f"Look-ahead : {len(df_la)} candles  ✓")
print(f"v_daily_avg: {v_daily_avg:.4f}")
print(f"dvr_lb MA range (excl. NaN): [{np.nanmin(dvr_lb_ma):.4f}, {np.nanmax(dvr_lb_ma):.4f}]")

In [ ]:
# ── Border datetimes ──────────────────────────────────────────────────────────
def fmt_ms(ts_ms):
    return pd.Timestamp(int(ts_ms), unit="ms", tz="UTC").strftime("%Y-%m-%d %H:%M:%S UTC")

lb_first_ts = df_lb["ts"].astype("int64").iloc[0]
lb_last_ts  = df_lb["ts"].astype("int64").iloc[-1]
la_first_ts = df_la["ts"].astype("int64").iloc[0]
la_last_ts  = df_la["ts"].astype("int64").iloc[-1]

print("─" * 52)
print(f"  current_time       :  {DATETIME} UTC")
print(f"  last_candle (open) :  {fmt_ms(lb_last_ts)}")
print("─" * 52)
print(f"  look-back  start   :  {fmt_ms(lb_first_ts)}")
print(f"  look-back  end     :  {fmt_ms(lb_last_ts)}")
print(f"  look-ahead start   :  {fmt_ms(la_first_ts)}")
print(f"  look-ahead end     :  {fmt_ms(la_last_ts)}")
print("─" * 52)

In [ ]:
# ── Original VWAP charts (side-by-side, proportional x-axis, shared y) ────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

x_lb  = np.arange(LOOK_BACK)
x_la  = np.arange(LOOK_AHEAD)
ts_lb = pd.to_datetime(df_lb["ts"].astype("int64"), unit="ms", utc=True)
ts_la = pd.to_datetime(df_la["ts"].astype("int64"), unit="ms", utc=True)

vwap_all   = np.concatenate([df_lb["vwap"].values, df_la["vwap"].values])
v_margin   = (vwap_all.max() - vwap_all.min()) * 0.05
price_ylim = (vwap_all.min() - v_margin, vwap_all.max() + v_margin)

fig = plt.figure(figsize=(14, 3))
gs  = gridspec.GridSpec(1, 2, width_ratios=[LOOK_BACK, LOOK_AHEAD], wspace=0.04)
ax_lb_p = fig.add_subplot(gs[0])
ax_la_p = fig.add_subplot(gs[1])

ax_lb_p.plot(x_lb, df_lb["vwap"].values, color="#26a69a", linewidth=0.9)
ax_lb_p.set_ylim(*price_ylim)
ax_lb_p.set_xlim(-1, LOOK_BACK)
ax_lb_p.set_title(f"{ASSET} — Look-Back ({LOOK_BACK} candles)  [past]", fontsize=9)
ax_lb_p.tick_params(axis="y", labelsize=7)
step_lb = max(1, LOOK_BACK // 6)
ax_lb_p.set_xticks(x_lb[::step_lb])
ax_lb_p.set_xticklabels([ts_lb.iloc[i].strftime("%H:%M") for i in x_lb[::step_lb]], fontsize=6)

ax_la_p.plot(x_la, df_la["vwap"].values, color="#ef5350", linewidth=0.9)
ax_la_p.set_ylim(*price_ylim)
ax_la_p.set_xlim(-1, LOOK_AHEAD)
ax_la_p.set_title(f"{ASSET} — Look-Ahead ({LOOK_AHEAD} candles)  [future]", fontsize=9)
ax_la_p.yaxis.tick_right()
ax_la_p.yaxis.set_label_position("right")
ax_la_p.tick_params(axis="y", labelsize=7)
step_la = max(1, LOOK_AHEAD // 6)
ax_la_p.set_xticks(x_la[::step_la])
ax_la_p.set_xticklabels([ts_la.iloc[i].strftime("%H:%M") for i in x_la[::step_la]], fontsize=6)

plt.suptitle(f"VWAP  |  current_time: {DATETIME} UTC", fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Price normalization + time normalization + KDE ────────────────────────────

# ── Price normalization ───────────────────────────────────────────────────────
price_l = df_lb["vwap"].values[-1]

raw_lb = (df_lb["vwap"].values - price_l) / price_l
raw_la = (df_la["vwap"].values - price_l) / price_l

if NORM_MODE == "clip":
    scaled_lb = np.clip(K * raw_lb, -1.0, 1.0)
    scaled_la = np.clip(K * raw_la, -1.0, 1.0)
elif NORM_MODE == "tanh":
    scaled_lb = np.tanh(K * raw_lb)
    scaled_la = np.tanh(K * raw_la)
else:
    raise ValueError(f"Unknown NORM_MODE: {NORM_MODE!r}")

# ── Time normalization ────────────────────────────────────────────────────────
t    = np.arange(LOOK_BACK) / LOOK_BACK
t_la = (LOOK_BACK + np.arange(LOOK_AHEAD)) / LOOK_BACK

# ── Volume for plotting (linear or log) ──────────────────────────────────────
if PLOT_VOLUME_LOGARITHM:
    v_lb_plot  = np.log(np.maximum(v_lb_norm, 1e-12))
    v_la_plot  = np.log(np.maximum(v_la_norm, 1e-12))
    vol_ylabel = "log(norm vol)"
else:
    v_lb_plot  = v_lb_norm
    v_la_plot  = v_la_norm
    vol_ylabel = "norm vol"

# ── KDE: volume-weighted histogram + kernel convolution ──────────────────────
if KDE_IGNORE_BORDERS:
    border_mask = (scaled_lb > -1.0) & (scaled_lb < 1.0)
    kde_prices  = scaled_lb[border_mask]
    kde_weights = v_lb_norm[border_mask]
    n_excluded  = (~border_mask).sum()
else:
    kde_prices  = scaled_lb
    kde_weights = v_lb_norm
    n_excluded  = 0

counts, bin_edges = np.histogram(
    kde_prices,
    bins=BINS_COUNT,
    range=(-1.0, 1.0),
    weights=kde_weights,
)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width   = bin_edges[1] - bin_edges[0]

def make_kernel(kernel_type, bandwidth):
    x = np.arange(-bandwidth, bandwidth + 1, dtype=float)
    if kernel_type == "Triangular":
        k = np.maximum(1.0 - np.abs(x) / bandwidth, 0.0)
    elif kernel_type == "Epanechnikov":
        k = np.maximum(1.0 - (x / bandwidth) ** 2, 0.0)
    elif kernel_type == "Uniform":
        k = np.ones(len(x))
    else:
        raise ValueError(f"Unknown kernel: {kernel_type!r}")
    k /= k.sum()
    return k

kernel_arr = make_kernel(KERNEL, BANDWIDTH)
kde        = np.convolve(counts, kernel_arr, mode="same")

print(f"price_l        = {price_l:.6f}")
print(f"k              = {K}   (ratio {1/K:.4f} → 1.0)")
print(f"t[0]           = {t[0]:.6f}   first look-back candle")
print(f"t[-1]          = {t[-1]:.6f}   last look-back candle (last_candle)")
print(f"current_time   = 1.000000   (t=1.0, not in array)")
print(f"t_la[0]        = {t_la[0]:.6f}   first look-ahead candle (opens at current_time)")
print(f"t_la[-1]       = {t_la[-1]:.6f}   last look-ahead candle")
print(f"scaled_lb      [{scaled_lb.min():.4f}, {scaled_lb.max():.4f}]")
print(f"scaled_la      [{scaled_la.min():.4f}, {scaled_la.max():.4f}]")
print(f"KDE borders excluded: {n_excluded} candles")
print(f"KDE total mass = {kde.sum():.4f}  (raw histogram mass = {counts.sum():.4f})")

In [ ]:
# ── KDE peak detection (scipy) ────────────────────────────────────────────────
from scipy.signal import find_peaks, peak_prominences

PEAK_COLORS = ["red", "orange", "yellow"]

pos_mask   = bin_centers >= 0
neg_mask   = bin_centers <  0

pos_kde    = kde[pos_mask]
pos_prices = bin_centers[pos_mask]
neg_kde    = kde[neg_mask]
neg_prices = bin_centers[neg_mask]

def top_kde_peaks(kde_series, prices, distance, n=3):
    peaks, _ = find_peaks(kde_series, distance=distance)
    if len(peaks) == 0:
        return np.array([]), np.array([])
    proms = peak_prominences(kde_series, peaks)[0]
    order = np.argsort(proms)[::-1][:n]
    return prices[peaks[order]], proms[order]

pos_peak_prices, pos_peak_proms = top_kde_peaks(pos_kde, pos_prices, distance=BANDWIDTH)
neg_peak_prices, neg_peak_proms = top_kde_peaks(neg_kde, neg_prices, distance=BANDWIDTH)

print("Positive-price KDE peaks  (top 3 by prominence):")
for i, (price, prom) in enumerate(zip(pos_peak_prices, pos_peak_proms)):
    print(f"  rank {i+1}  color={PEAK_COLORS[i]:6s}  price={price:+.4f}  prominence={prom:.4f}")
if len(pos_peak_prices) == 0:
    print("  (no peaks found)")

print("Negative-price KDE peaks  (top 3 by prominence):")
for i, (price, prom) in enumerate(zip(neg_peak_prices, neg_peak_proms)):
    print(f"  rank {i+1}  color={PEAK_COLORS[i]:6s}  price={price:+.4f}  prominence={prom:.4f}")
if len(neg_peak_prices) == 0:
    print("  (no peaks found)")

In [ ]:
# ── KDE + normalized VWAP + DVR MA + volume charts ────────────────────────────
# Layout (3 rows × 3 cols):
#   row 0: [KDE]  [look-back norm VWAP]    [look-ahead norm VWAP]
#   row 1: [    ] [look-back DVR MA]       [look-ahead DVR MA]
#   row 2: [    ] [look-back norm vol]     [look-ahead norm vol]
# KDE occupies row 0 only; rows 1 and 2 share the time x-axis with row 0.
# x-tick labels appear on row 2 only.

fig = plt.figure(figsize=(14, 6))
gs  = gridspec.GridSpec(
    3, 3,
    width_ratios=[LOOK_AHEAD, LOOK_BACK, LOOK_AHEAD],
    height_ratios=[3, 1, 1],
    hspace=0.06,
    wspace=0.04,
)

ax_kde    = fig.add_subplot(gs[0, 0])
ax_lb_n   = fig.add_subplot(gs[0, 1], sharey=ax_kde)
ax_la_n   = fig.add_subplot(gs[0, 2], sharey=ax_kde)
ax_lb_dvr = fig.add_subplot(gs[1, 1], sharex=ax_lb_n)
ax_la_dvr = fig.add_subplot(gs[1, 2], sharex=ax_la_n)
ax_lb_vol = fig.add_subplot(gs[2, 1], sharex=ax_lb_n)
ax_la_vol = fig.add_subplot(gs[2, 2], sharex=ax_la_n)
# gs[1,0] and gs[2,0] intentionally empty (below KDE)

# ── KDE ───────────────────────────────────────────────────────────────────────
ax_kde.barh(
    bin_centers, kde,
    height=bin_width * 0.95,
    color="#26a69a", alpha=0.85, linewidth=0,
)
ax_kde.axhline(0,  color="#aaaaaa", linewidth=0.5, linestyle="--")
ax_kde.axhline( 1, color="#ff9800", linewidth=0.4, linestyle=":")
ax_kde.axhline(-1, color="#ff9800", linewidth=0.4, linestyle=":")
ax_kde.set_ylim(-1.08, 1.08)
ax_kde.set_ylabel("scaled vwap", fontsize=8)
ax_kde.set_xlabel("vol (daily avg units)", fontsize=7)
border_note = "  (±1 excl.)" if KDE_IGNORE_BORDERS else ""
ax_kde.set_title(
    f"Vol-Weighted KDE\n(kernel={KERNEL}, bw={BANDWIDTH}{border_note})",
    fontsize=9,
)
ax_kde.tick_params(labelsize=7)

# ── Look-back normalized VWAP ─────────────────────────────────────────────────
ax_lb_n.plot(t, scaled_lb, color="#26a69a", linewidth=0.8)
ax_lb_n.axhline(0,  color="#aaaaaa", linewidth=0.5, linestyle="--")
ax_lb_n.axhline( 1, color="#ff9800", linewidth=0.4, linestyle=":")
ax_lb_n.axhline(-1, color="#ff9800", linewidth=0.4, linestyle=":")
ax_lb_n.set_title(f"Look-Back — normalized VWAP  (k={K}, mode={NORM_MODE})", fontsize=9)
plt.setp(ax_lb_n.get_yticklabels(), visible=False)
plt.setp(ax_lb_n.get_xticklabels(), visible=False)

# ── Look-ahead normalized VWAP ────────────────────────────────────────────────
ax_la_n.plot(t_la, scaled_la, color="#ef5350", linewidth=0.8)
ax_la_n.axhline(0,  color="#aaaaaa", linewidth=0.5, linestyle="--")
ax_la_n.axhline( 1, color="#ff9800", linewidth=0.4, linestyle=":")
ax_la_n.axhline(-1, color="#ff9800", linewidth=0.4, linestyle=":")
ax_la_n.set_title("Look-Ahead — normalized VWAP  (same price_l anchor)", fontsize=9)
ax_la_n.yaxis.tick_right()
ax_la_n.yaxis.set_label_position("right")
ax_la_n.set_ylabel("scaled vwap", fontsize=8)
ax_la_n.tick_params(labelsize=7)
plt.setp(ax_la_n.get_xticklabels(), visible=False)

# ── KDE peak lines on shared-y axes ──────────────────────────────────────────
for i, price in enumerate(pos_peak_prices):
    for ax in [ax_kde, ax_lb_n, ax_la_n]:
        ax.axhline(price, color=PEAK_COLORS[i], linewidth=0.9, linestyle="-", alpha=0.85, zorder=4)
for i, price in enumerate(neg_peak_prices):
    for ax in [ax_kde, ax_lb_n, ax_la_n]:
        ax.axhline(price, color=PEAK_COLORS[i], linewidth=0.9, linestyle="-", alpha=0.85, zorder=4)

# ── DVR MA charts ─────────────────────────────────────────────────────────────
def plot_dvr_ma(ax, t_axis, dvr_ma, y_right=False):
    valid = ~np.isnan(dvr_ma)
    tv, mv = t_axis[valid], dvr_ma[valid]
    ax.plot(tv, mv, color="#90caf9", linewidth=0.8, zorder=3)
    ax.fill_between(tv, mv, 0,
                    where=(mv >= 0), color="#26a69a", alpha=0.4, linewidth=0)
    ax.fill_between(tv, mv, 0,
                    where=(mv <  0), color="#ef5350", alpha=0.4, linewidth=0)
    ax.axhline(0, color="#aaaaaa", linewidth=0.5, linestyle="--")
    ax.set_ylim(-1.08, 1.08)
    ax.set_ylabel(f"DVR MA({DELTA_VOLUME_RATIO_PERIOD})", fontsize=7)
    ax.tick_params(axis="y", labelsize=6)
    plt.setp(ax.get_xticklabels(), visible=False)
    if y_right:
        ax.yaxis.tick_right()
        ax.yaxis.set_label_position("right")

plot_dvr_ma(ax_lb_dvr, t,    dvr_lb_ma)
plot_dvr_ma(ax_la_dvr, t_la, dvr_la_ma, y_right=True)

# ── Volume charts ─────────────────────────────────────────────────────────────
vol_all    = np.concatenate([v_lb_plot, v_la_plot])
vol_margin = (vol_all.max() - vol_all.min()) * 0.08
vol_ylim   = (vol_all.min() - vol_margin, vol_all.max() + vol_margin)

bar_width_lb = t[1]    - t[0]    if len(t)    > 1 else 1 / LOOK_BACK
bar_width_la = t_la[1] - t_la[0] if len(t_la) > 1 else 1 / LOOK_BACK

ax_lb_vol.bar(t, v_lb_plot, width=bar_width_lb, color="#26a69a", linewidth=0, alpha=0.8)
ax_lb_vol.set_ylim(*vol_ylim)
ax_lb_vol.set_ylabel(vol_ylabel, fontsize=7)
ax_lb_vol.tick_params(axis="y", labelsize=6)
if PLOT_VOLUME_LOGARITHM:
    ax_lb_vol.axhline(0, color="#aaaaaa", linewidth=0.4, linestyle="--")
step_lb_v   = max(1, LOOK_BACK // 6)
tick_idx_lb = np.arange(0, LOOK_BACK, step_lb_v)
ax_lb_vol.set_xticks(t[tick_idx_lb])
ax_lb_vol.set_xticklabels(
    [ts_lb.iloc[i].strftime("%H:%M") for i in tick_idx_lb], fontsize=6
)

ax_la_vol.bar(t_la, v_la_plot, width=bar_width_la, color="#ef5350", linewidth=0, alpha=0.8)
ax_la_vol.set_ylim(*vol_ylim)
ax_la_vol.yaxis.tick_right()
ax_la_vol.yaxis.set_label_position("right")
ax_la_vol.set_ylabel(vol_ylabel, fontsize=7)
ax_la_vol.tick_params(axis="y", labelsize=6)
if PLOT_VOLUME_LOGARITHM:
    ax_la_vol.axhline(0, color="#aaaaaa", linewidth=0.4, linestyle="--")
step_la_v   = max(1, LOOK_AHEAD // 6)
tick_idx_la = np.arange(0, LOOK_AHEAD, step_la_v)
ax_la_vol.set_xticks(t_la[tick_idx_la])
ax_la_vol.set_xticklabels(
    [ts_la.iloc[i].strftime("%H:%M") for i in tick_idx_la], fontsize=6
)

plt.suptitle(
    f"KDE + Normalized VWAP + DVR MA + Volume  |  price_l={price_l:.4f}   k={K}   current_time={DATETIME} UTC",
    fontsize=10,
    y=1.01,
)
plt.tight_layout()
plt.show()